# 06 · Scientific plots: formulas and simulations

**Session 6 — Branch and Plot** · about 30 minutes

**You will learn to**
- plot explicit, parametric and implicit functions and 3D surfaces
- check a formula numerically before trusting its plot
- simulate the Kuramoto model and plot its order parameter
- sketch a figure before writing (or prompting) any code

**Data:** none (formulas and simulations).

**Outline**
0. Before you start (no AI)
1. Formulas
2. Simulation: the Kuramoto model
3. From your sketch to your data

> **AI guardrails** — *Think first · Own it · Verify · Disclose · Protect.* AI tools are allowed in work time, but answer the **No-AI** questions on paper first, and make every ✅ self-check pass with code you can explain line by line.

## 0. Before you start (no AI)

Sketch on paper, with axes and labels:

1. the density of a positive random variable with a long right tail;
2. how the synchrony *r* (0 = incoherent, 1 = synchronised) of coupled oscillators should change
   as the coupling *K* grows.

Keep both sketches: you will compare them with the plots below.

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import odeint, quad

In [ ]:
from pathlib import Path

DATA = Path("../data")
OUTPUTS = Path("../outputs")
OUTPUTS.mkdir(exist_ok=True)
assert DATA.exists(), "Start Jupyter from the repository folder: notebooks/ must sit next to data/."

## 1. Formulas

### Explicit: the inverse Gaussian (Wald) distribution

$$f(x;\mu,\lambda) = \sqrt{\frac{\lambda}{2\pi x^3}}\,\exp\!\left(-\frac{\lambda (x-\mu)^2}{2\mu^2 x}\right), \quad x>0$$

### Verify the AI

An assistant wrote the function below. Before plotting, check the one property every density
must have: it integrates to 1.

In [ ]:
def wald_ai(x, lam, mu):
    return lam / (2 * np.pi * x**3) * np.exp(-lam * (x - mu) ** 2 / (2 * mu**2 * x))


area, _ = quad(wald_ai, 0, np.inf, args=(3, 1))
print(f"integral = {area:.3f}")

### TASK I — fix the density

Write `wald(x, lam, mu)` following the formula above, check that it integrates to 1 for
`lam=3, mu=1`, then plot it for every combination of `lam in [0.3, 1, 3]` and `mu in [1, 3]`
on `x = np.linspace(0.001, 3, 1000)`, with a legend and labelled axes.

In [ ]:
# TASK I — your code here
def wald(x, lam, mu):
    ...

In [ ]:
# ✅ Self-check TASK I
assert abs(quad(wald, 0, np.inf, args=(3, 1))[0] - 1) < 1e-6, "The density must integrate to 1."
assert abs(quad(wald, 0, np.inf, args=(0.3, 3))[0] - 1) < 1e-4
print("✅ TASK I passed")

### Parametric: a circle in one call

In [ ]:
phi = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.plot(np.cos(phi), np.sin(phi))
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

### Implicit: $x\sin x = y^2 \cos y$

No obvious parametrisation? Plot the zero level of $F(x, y) = x\sin x - y^2\cos y$ as a contour.

In [ ]:
xx, yy = np.meshgrid(np.linspace(-np.pi / 2, np.pi / 2, 600), np.linspace(-np.pi / 2, np.pi / 2, 600))
fig, ax = plt.subplots(figsize=(4, 4))
ax.contour(xx, yy, xx * np.sin(xx) - yy**2 * np.cos(yy), levels=[0])
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

### 3D surface: $R(\theta,\phi) = \cos(\phi^2)$

In [ ]:
theta, phi = np.meshgrid(np.linspace(0, 2 * np.pi, 100), np.linspace(0, np.pi, 100))
R = np.cos(phi**2)
X, Y, Z = R * np.sin(phi) * np.cos(theta), R * np.sin(phi) * np.sin(theta), R * np.cos(phi)

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection="3d")
ax.plot_surface(X, Y, Z, cmap="viridis", alpha=0.8)
ax.set(xlabel="x", ylabel="y", zlabel="z")
plt.show()

## 2. Simulation: the Kuramoto model

$N$ oscillators with natural frequencies $\omega_i \sim \mathcal N(0, 1)$ and global coupling $K$:

$$\dot\theta_i = \omega_i + \frac{K}{N}\sum_{j=1}^{N} \sin(\theta_j - \theta_i) = \omega_i + K\,r\,\sin(\psi - \theta_i),
\qquad r\,e^{i\psi} = \frac1N\sum_j e^{i\theta_j}.$$

The second form (mean field) costs $O(N)$ instead of $O(N^2)$ per step.

In [ ]:
def kuramoto(K, N=200, t_max=60.0, dt=0.05, seed=42):
    """Integrate the Kuramoto model; return times, phases and the order parameter r(t)."""
    rng = np.random.default_rng(seed)
    omega = rng.normal(0, 1, N)
    theta0 = rng.uniform(0, 2 * np.pi, N)

    def rhs(theta, t):
        z = np.exp(1j * theta).mean()
        return omega + K * np.abs(z) * np.sin(np.angle(z) - theta)

    t = np.arange(0, t_max, dt)
    theta = odeint(rhs, theta0, t)
    z = np.exp(1j * theta).mean(axis=1)
    return t, z


t, z = kuramoto(K=2.5)
fig = plt.figure(figsize=(9, 3.6), layout="constrained")
ax1 = fig.add_subplot(1, 2, 1, projection="polar")
ax1.plot(np.angle(z), np.abs(z), color="grey", lw=0.8)
ax1.plot(np.angle(z[:20]), np.abs(z[:20]), color="red", lw=2, label="start")
ax1.plot(np.angle(z[-20:]), np.abs(z[-20:]), color="orange", lw=2, label="end")
ax1.set_rmax(1.05)
ax1.legend(loc="center left", bbox_to_anchor=(1.1, 0.5), frameon=False)
ax1.set_title("Order parameter $r e^{i\\psi}$")
ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(t, np.abs(z))
ax2.set(xlabel="time", ylabel="r(t)", ylim=(0, 1), title="Synchronisation, K = 2.5")
plt.show()

### TASK II — find the synchronisation transition

1. For `K` from 0 to 4 in steps of 0.25, run `kuramoto(K)` and store the **steady-state** synchrony
   `r_inf` = mean of `|z|` over the last 20% of the time.
2. Plot `r_inf` against `K` (markers and line), with labelled axes.
3. Add a vertical line at the theoretical critical coupling for $\mathcal N(0,1)$ frequencies,
   $K_c = 2 / (\pi g(0)) = 2\sqrt{2\pi}/\pi \approx 1.60$.

*Compare with your sketch from section 0.* Store the results in `Ks` and `r_inf` (arrays).

In [ ]:
# TASK II — your code here
Ks, r_inf = None, None

In [ ]:
# ✅ Self-check TASK II
assert Ks is not None and len(Ks) == 17 and len(r_inf) == 17
assert r_inf[0] < 0.2, "Without coupling the oscillators should stay incoherent."
assert r_inf[-1] > 0.75, "Strong coupling should synchronise them."
assert 1.25 <= Ks[np.argmax(r_inf > 0.4)] <= 2.5, "The jump should happen near K_c."
print("✅ TASK II passed")

## 3. From your sketch to your data

An AI assistant can write plotting code in seconds; it cannot know which comparison matters to your
story. In your group repository, on a branch:

1. sketch on paper the first figure of your project (x, y, colour, panels) and write the one sentence it
   should prove;
2. write the code (with or without AI);
3. compare the result with the sketch, not the other way round; open a pull request.

**Next:** `07_plotting_ecosystem.ipynb` (Session 8, homework).